## Notebook used for the extraction of the BHL data, we decided to add threads to make the extraction a lot quickier
---

Proyect configuration, including imports, variables initiation:
* We are using the INBIO CSV
* We are working with th API v3 off BHL, using API Keys that the site provides for everyone
---

In [8]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import re
from tqdm.notebook import tqdm  #Used for Progress Bars
import time #Used for the time sleep at the moment of retrieving data
from google.colab import files
import threading #Used for the threads implementation
# API configuration
API_KEY = "4aee48b9-514f-45b9-9396-67c38ba7c04c" #
API_URL = "https://www.biodiversitylibrary.org/api3"
# CSV URL
csv_url = "https://drive.google.com/file/d/1nnd7rRLAI_GaOnVanZS6u4F69B2AmzVF/view?usp=sharing"

def read_species_data(url):
    """Makes a DataFrame with the data located in the csv that recibes."""
    file_id = url.split('/')[-2]
    path = f'https://drive.google.com/uc?export=download&id={file_id}'
    return pd.read_csv(path, sep='|')

###Methods used
####Methods given by the BHL API to extract data:
* GetNameMetadata (Api Method that let us extract all the pages with info about one specie, the Api´s response is saved in a list)
* GetPageMetadata(Api Method that let us extract the OCR text in a page, the Api´s response is cleaned then saved in a dataframe)
* GetItemMetadata(Api Method that let us extract the metadata of a page, the Api´s response is saved in the dataframe for future use)

https://www.biodiversitylibrary.org/docs/api3.html#methods

API´s Documentation

---

In [9]:
failed_pages = []
def get_page_metadata(page_id):
    """Extracts the metadata of the page with the ID that matches the argument."""
    if(page_id in failed_pages): return None
    params = {
        "op": "GetPageMetadata",
        "pageid": page_id,
        "ocr": "t",
        "names": "t",
        "format": "xml",
        "apikey": API_KEY
    }
    try:
        response = requests.get(API_URL, params=params, timeout=25)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Error extracting the data for this page: {page_id}: {str(e)}")
        failed_pages.append(page_id)
        return None

def get_name_metadata(name):
    """Extracts the metadata of the species with the same name that the argument."""
    params = {
        "op": "GetNameMetadata",
        "name": name,
        "format": "xml",
        "apikey": API_KEY
    }
    try:
        response = requests.get(API_URL, params=params, timeout=25)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Error extracting the data for this species {name}: {str(e)}")
        return None


failed_items = []
def get_item_metadata(item_ID):
    """Extracts the metadata of the item with the ID that matches the argument."""
    if(item_ID in failed_items): return None
    params = {
        "op": "GetItemMetadata",
        "id": item_ID,
        "idtype": "bhl",
        "pages": "f",  #Data already retrieved, doesn´t need to ask it again
        "ocr": "f",
        "parts": "f",
        "format": "xml",
        "apikey": API_KEY
    }
    try:
        response = requests.get(API_URL, params=params, timeout=25)
        response.raise_for_status()
        return response.text
    except Exception as e:
        failed_items.append(item_ID)
        print(f"Error extracting the data for this item: {item_ID}: {str(e)}")
        return None



###Methods used for extracting data
####All of the xmls are responses of the API´S methos explained before
---

In [10]:
def conseguirListaPaginas(xml_data):
    """Parses the response of the Api´s method GetNameMetadata"""
    if not xml_data:
        return []
    try:
        root = ET.fromstring(xml_data)
        return [page_id.text for page_id in root.findall('.//PageID')]
    except Exception as e:
        print(f"Error parsing a XML: {str(e)}")
        return []

def extract_idItem_metadata(xml_data):
    """Extracts important metadata in the response of the Api´s method GetItemMetadata"""
    if not xml_data:
        return {}
    try:
        root = ET.fromstring(xml_data)
        return {
            'Source': root.find('.//Source').text if root.find('.//Source') is not None else None,
            'Source_ID': root.find('.//SourceIdentifier').text if root.find('.//SourceIdentifier') is not None else None,
            'Institution': root.find('.//HoldingInstitution').text if root.find('.//HoldingInstitution') is not None else None,
            'Language': root.find('.//Language').text if root.find('.//Language') is not None else None,
            'Rights': root.find('.//Rights').text if root.find('.//Rights') is not None else None,
            'Copyright': root.find('.//CopyrightStatus').text if root.find('.//CopyrightStatus') is not None else None,
        }
    except Exception as e:
        print(f"Error extracting metadata in a XML: {str(e)}")
        return {}


def extract_page_metadata(xml_data):
    """Parses important info about a page in a XML then returns it"""
    if not xml_data:
        return {}
    try:
        root = ET.fromstring(xml_data)
        return {
            'Volume': root.find('.//Volume').text if root.find('.//Volume') is not None else None,
            'Year': root.find('.//Year').text if root.find('.//Year') is not None else None,
        }
    except Exception as e:
        print(f"Error extracting metadata in a XML: {str(e)}")
        return {}

###Method for cleaning the text
####We prioritize error in the OCR, in other notebook in the same github we finish the cleaning.
---

In [11]:
def clean_paragraph(text):
    if not text:
        return ""

    replacements = [
        ('“', '"'),
        ('”', '"'),
        ('‘', "'"),
        ('’', "'"),
        ('—', '-'),
        ('–', '-'),
        ('…', '...'),
        ("- ", ""),
    ]

    for old, new in replacements:
        text = text.replace(old, new)

    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text if text else ""

##Methods for the creation of the CSV with all the data


In [12]:
def process_species_subset(csv_url, n=None, idioma="Both"):
    """
    Process the species in a subset:
    - If N is given: process the first N species
    - If N is None(Default in the function): Only process the first species
    """
    df = read_species_data(csv_url)
    if df.empty:
        return pd.DataFrame()
    if n:
        # Takes the first N species
        target_species = df["default_name"].unique().tolist()[:n]
    else:
        # Takes only one spcecies
        target_species = [df.iloc[0]["default_name"]]

    return _process_species_list(target_species, idioma)

def process_species_range(csv_url, start_index=0, end_index=None, idioma="Both"):
    """
    Extracts the species in the csv who are in the range given:
    - start_index: Where it will start, default 0
    - end_index: Where it will end, if None it will take all the species
    """
    df = read_species_data(csv_url)
    if df.empty:
        return pd.DataFrame()

    all_species = df["default_name"].unique().tolist()

    # end_index change if it is bigger that the lenght of the list of species
    end_index = min(end_index, len(all_species)) if end_index is not None else len(all_species)

    target_species = all_species[start_index:end_index]

    return _process_species_list(target_species, idioma)

def process_all_species(csv_url, idioma="Ambas"):
    """As is name says, process ALL species in the given csv."""
    df = read_species_data(csv_url)
    if df.empty:
        return pd.DataFrame()

    unique_species = df["default_name"].unique().tolist()
    print(f"Processing al the species, total: {len(unique_species)}")

    return _process_species_list(unique_species, idioma)

def _process_species_list(species_list, lenguaje):
    """Main method to process a list with species, it also recieves the language to extract."""
    data_rows = []

    for species in tqdm(species_list, desc="Proccesing the data"):
        # Pause time so the API KEY doesn´t get banned
        time.sleep(0.35)

        name_metadata = get_name_metadata(species)
        if not name_metadata:
            continue

        page_ids_preset = conseguirListaPaginas(name_metadata)
        if not page_ids_preset:
            continue
        set_ids = set(page_ids_preset)
        page_ids = list(set_ids)

        num_errores = 0

        for page_id in page_ids:

            if(num_errores == 50): break
            #If a species present a lot of problems it is better to stop trying and go to the next
            page_xml = get_page_metadata(page_id)
            if not page_xml:
                num_errores+=1
                continue

            try:
                root = ET.fromstring(page_xml)
            except ET.ParseError:
                num_errores+=1
                continue

            page_meta = extract_page_metadata(page_xml)
            ocr_text = root.find('.//OcrText').text if root.find('.//OcrText') is not None else ""

            item_id = root.find('.//ItemID').text if root.find('.//ItemID') is not None else ""
            item_xml = get_item_metadata(item_id)
            if not item_xml:
                num_errores+=1
                continue

            try:
                root = ET.fromstring(item_xml)
            except ET.ParseError:
                num_errores+=1
                continue

            idioma = root.find('.//Language').text if root.find('.//Language') is not None else ""
            if(lenguaje == "Spanish"):
              if(not(idioma == "Spanish")):
                continue
            if(lenguaje == "English"):
              if(not(idioma == "English")):
                continue
            if(lenguaje == "Both"):
              if(not((idioma == "English")or(idioma == "Spanish"))):
                continue
            item_meta = extract_idItem_metadata(item_xml)

            for para in [p.strip() for p in ocr_text.split("\n\n") if p.strip()]:
                cleaned_para = clean_paragraph(para)
                word_count = len(cleaned_para.split())

                if (word_count >= 50):
                    row = {
                        "Species": species,
                        "Text": cleaned_para,
                        "Page_id": page_id
                    }
                    row.update(page_meta)
                    row.update(item_meta)
                    data_rows.append(row)

    return pd.DataFrame(data_rows)

##Threads implementation
####In this case we have some APIS KEYS, we asked BHL API for this keys.
####The APIS KEYS are in a list for easy access in the moment we want to use them

---

In [13]:
API_KEYS = [
    "4aee48b9-514f-45b9-9396-67c38ba7c04c", #1
    "1fbc22e8-67c1-4191-b556-0d87e61d75c5", #2
    "7f2517fc-46d1-4745-985b-6947c417f294", #3
    "7ab0fd8f-a839-4e31-bf5e-bae52964d336", #4
    "2a7130a2-c590-4578-a1e0-91a67831d322", #5
    "e25f99af-c4c2-4ddf-a35e-bb491af48114", #6
    "1ca683b1-d689-4b20-a9da-2f5935de2ef8", #7
    "d4e7fe03-739d-4a25-9328-d7ea820ebb91", #8
    "3aa034a2-35e8-458c-8f6a-220abed9432f", #9
    "4f5f4c7e-3273-49a7-88c1-f44aaa17b6fa", #10
    "59a24795-5c34-432a-9f94-0c7d657fd3cf"  #11
]
print(len(API_KEYS))
class ThreadedSpeciesProcessor:
    def __init__(self, csv_url, total_species, idioma="Both", num_threads=None):

        self.csv_url = csv_url
        self.total_species = total_species
        self.idioma = idioma
        self.num_threads = num_threads if num_threads else len(API_KEYS)
        self.results = [None] * self.num_threads
        self.api_keys = API_KEYS[:self.num_threads]  # Use only needed keys

    def worker(self, thread_idx, species_chunk):
        global API_KEY
        API_KEY = self.api_keys[thread_idx]

        thread_dfs = []
        with tqdm(total=len(species_chunk),
                 desc=f"Thread {thread_idx+1}/{self.num_threads} (API {API_KEY[-4:]})",
                 position=thread_idx,
                 leave=True) as pbar:

            for species in species_chunk:
                time.sleep(0.35)  # Rate limiting
                try:
                    species_df = _process_species_list([species], self.idioma)
                    if species_df is not None and not species_df.empty:
                        thread_dfs.append(species_df)
                except Exception as e:
                    tqdm.write(f"Thread {thread_idx+1} error on {species}: {str(e)}")
                finally:
                    pbar.update(1)

        if thread_dfs:
            self.results[thread_idx] = pd.concat(thread_dfs, ignore_index=True)

    def process(self):
        df = read_species_data(self.csv_url)
        if df.empty:
            raise ValueError("No species data found in the CSV")

        unique_species = df["default_name"].unique().tolist()[:self.total_species]

        chunk_size = len(unique_species) // self.num_threads
        chunks = []
        for i in range(self.num_threads):
            start = i * chunk_size
            end = (i + 1) * chunk_size if i < self.num_threads - 1 else len(unique_species)
            chunks.append(unique_species[start:end])

        threads = []
        for i in range(self.num_threads):
            thread = threading.Thread(target=self.worker, args=(i, chunks[i]))
            threads.append(thread)
            thread.start()

        for thread in threads:
            thread.join()

        valid_results = [df for df in self.results if df is not None and not df.empty]
        if not valid_results:
            raise ValueError("No valid results produced. Check API keys and filters")
        return pd.concat(valid_results, ignore_index=True)

11


## Call of the processor constructor with the desired arguments, then we use the processor class methods
* if the function is called with the number of threads variable it will only use the first num_threads Api Keys
* if called without that parameter it will use all of the api keys in the API_KEYS list

In [ ]:
processor = ThreadedSpeciesProcessor(csv_url, 500, "Both")
df = processor.process()

Thread 2/11 (API 75c5):   0%|          | 0/45 [00:00<?, ?it/s]

Thread 4/11 (API d336):   0%|          | 0/45 [00:00<?, ?it/s]

Thread 3/11 (API f294):   0%|          | 0/45 [00:00<?, ?it/s]

Thread 1/11 (API c04c):   0%|          | 0/45 [00:00<?, ?it/s]

Thread 7/11 (API 2ef8):   0%|          | 0/45 [00:00<?, ?it/s]

...

Proccesing the data:   0%|          | 0/1 [00:00<?, ?it/s]

Proccesing the data:   0%|          | 0/1 [00:00<?, ?it/s]

Proccesing the data:   0%|          | 0/1 [00:00<?, ?it/s]

Proccesing the data:   0%|          | 0/1 [00:00<?, ?it/s]

Proccesing the data:   0%|          | 0/1 [00:00<?, ?it/s]

###The download of the new csv

In [15]:
df.to_csv("EspeciesReadyToclean.csv", encoding='utf-8-sig', index=False)
files.download("EspeciesReadyToclean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>